In [2]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 40)

In [3]:
def find_fase1_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if candidate.name == 'fase_1_diagnostico':
            return candidate
    raise RuntimeError('Nao encontrei fase_1_diagnostico a partir deste notebook')

CURRENT_DIR = Path.cwd().resolve()
FASE1_ROOT = find_fase1_root(CURRENT_DIR)
DADOS_INICIAIS = FASE1_ROOT / 'dados' / 'dados_iniciais'
MANIFESTO_CLASSIFICADO = FASE1_ROOT / 'dados' / 'manifestos_final' / 'manifest_arquivos_classificados.csv'

{
    'fase1_root': str(FASE1_ROOT),
    'dados_iniciais_existe': DADOS_INICIAIS.exists(),
    'manifesto_existe': MANIFESTO_CLASSIFICADO.exists(),
}

{'fase1_root': '/home/wilson/Maringa/fase_1_diagnostico',
 'dados_iniciais_existe': True,
 'manifesto_existe': True}

In [4]:
manifest_df = pd.read_csv(MANIFESTO_CLASSIFICADO)
manifest_df[['pasta_raiz_canonica', 'nome_arquivo', 'dominio', 'forno', 'granularidade', 'observacoes_granularidade']].head(5)

,pasta_raiz_canonica,nome_arquivo,dominio,forno,granularidade,observacoes_granularidade
0,Consumo Fornos,2018_F2_Consumo.csv,consumo_fornos,F2,nao_definida,Sem indicacao clara para consumo_fornos
1,Consumo Fornos,2018_F3_Consumo.csv,consumo_fornos,F3,nao_definida,Sem indicacao clara para consumo_fornos
2,Consumo Fornos,2018_F4_Consumo.csv,consumo_fornos,F4,nao_definida,Sem indicacao clara para consumo_fornos
3,Consumo Fornos,2018_F5_Consumo.csv,consumo_fornos,F5,nao_definida,Sem indicacao clara para consumo_fornos
4,Consumo Fornos,2019_F1_Consumo.csv,consumo_fornos,F1,nao_definida,Sem indicacao clara para consumo_fornos


In [5]:
def sample_by_pasta(pasta: str, n: int = 5):
    filtro = manifest_df['pasta_raiz_canonica'] == pasta
    cols = ['nome_arquivo', 'forno', 'granularidade', 'observacoes_granularidade']
    return manifest_df.loc[filtro, cols].head(n)

sample_by_pasta('Consumo Fornos', n=3)

,nome_arquivo,forno,granularidade,observacoes_granularidade
0,2018_F2_Consumo.csv,F2,nao_definida,Sem indicacao clara para consumo_fornos
1,2018_F3_Consumo.csv,F3,nao_definida,Sem indicacao clara para consumo_fornos
2,2018_F4_Consumo.csv,F4,nao_definida,Sem indicacao clara para consumo_fornos


In [6]:
pasta_resumo = {p.name: len(list(p.glob('*.csv'))) for p in DADOS_INICIAIS.iterdir() if p.is_dir()}
dict(sorted(pasta_resumo.items()))

{'Consumo Fornos': 39,
 'Corridas': 40,
 'Dicionário': 0,
 'Eletrodo': 1,
 'Informações Diária': 40,
 'Supervisorio Forno 4': 2,
 'Supervisorio Forno 5': 2}

In [7]:
CONSUMO_DIR = DADOS_INICIAIS / 'Consumo Fornos'
INFO_DIARIA_DIR = DADOS_INICIAIS / 'Informações Diária'
SUPERV4_DIR = DADOS_INICIAIS / 'Supervisorio Forno 4'
SUPERV5_DIR = DADOS_INICIAIS / 'Supervisorio Forno 5'

def preview_csv(path: Path, *, sep: str = ';', encoding: str = 'latin1', nrows: int = 5, **kwargs):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, sep=sep, encoding=encoding, nrows=nrows, **kwargs)

preview_csv(CONSUMO_DIR / '2018_F2_Consumo.csv', nrows=3)

,"Forno,Data,Folha,Bal.,Silo,Código,Descrição,Umid. (%),Peso Úmido (kg/t),Lote2,Mn (%),Fe (%),SiO2 (%),CaO (%),MgO (%),Al2O3 (%),K2O (%),Na2O (%),BaO (%),P (%),TiO2 (%),C (%),C. Fixo (%),Cinzas (%),Voláteis (%),Consumo Bruto (t)"
0,"2,1/1/2018,Folha 01,1,7,13100004,Buritirama ,1..."
1,"2,1/1/2018,Folha 01,1,6,13100028,Sereno,5.00,1..."
2,"2,1/1/2018,Folha 01,1,5,13100001,Urucum ST,6.0..."


In [8]:
preview_csv(INFO_DIARIA_DIR / '2018_F1_Inf.Diario.csv', nrows=3)

,"Base,Forno,Data_base,Liga,Comp_Teorico_Ele1,Comp_Teorico_Ele2,Comp_Teorico_Ele3,Comp_Real_Ele1,Comp_Real_Ele2,Comp_Real_Ele3,Const_Deslz_Ele1,Const_Deslz_Ele2,Const_Deslz_Ele3,Nivel_Pasta_Ele1,Nivel_Pasta_Ele2,Nivel_Pasta_Ele3,Carreg_Pasta_3T_Ele1,Carreg_Pasta_3T_Ele2,Carreg_Pasta_3T_Ele3,Carreg_Pasta_1T_Ele1,Carreg_Pasta_1T_Ele2,Carreg_Pasta_1T_Ele3,Carreg_Pasta_2T_Ele1,Carreg_Pasta_2T_Ele2,Carreg_Pasta_2T_Ele3,Carreg_Pasta_Total,Camisa_Sold_Ele1,Camisa_Sold_Ele2,Camisa_Sold_Ele3,IOF,Potencia_Media,Cons_Energ_Real,Cons_Energ_Prev,CEE,Fator_Carga,El. I (kA),El. II (kA),El. III (kA),El. I (V),El. II (V),El. III (V),CEC Rod,CEC Real,CF Leito,,,,,,,,"
0,"1-43101,1,1/1/2018,FeMnAC,2.79,3.55,2.97,2.46,..."
1,"1-43102,1,1/2/2018,FeMnAC,2.78,3.46,2.98,2.46,..."
2,"1-43103,1,1/3/2018,FeMnAC,2.75,3.40,2.98,2.46,..."


In [9]:
def describe_temporal_columns(path: Path, *, sample_rows: int = 200) -> dict:
    df = preview_csv(path, nrows=sample_rows)
    col_datas = [col for col in df.columns if 'data' in col.lower()]
    return {
        'arquivo': path.name,
        'total_colunas': len(df.columns),
        'colunas_data': col_datas,
    }

analises = [
    describe_temporal_columns(CONSUMO_DIR / '2018_F2_Consumo.csv'),
    describe_temporal_columns(INFO_DIARIA_DIR / '2018_F1_Inf.Diario.csv'),
]
pd.DataFrame(analises)

,arquivo,total_colunas,colunas_data
0,2018_F2_Consumo.csv,1,"[Forno,Data,Folha,Bal.,Silo,Código,Descrição,U..."
1,2018_F1_Inf.Diario.csv,1,"[Base,Forno,Data_base,Liga,Comp_Teorico_Ele1,C..."


## Proximos passos
- Ampliar a inspeção para pastas de Supervisório e Eletrodos.
- Atualizar `observacoes_granularidade` com pistas coletadas e reexecutar o `manifest_unificado`.
- Validar se os anos/meses inferidos batem com as colunas de data identificadas.